# Kafka producer -> Spark consumer

Needs Kafka: `make kafka-install`. The producer is a plain Python app (kafka-python)
that sends order lifecycles (CREATED -> PAID -> PACKED -> SHIPPED -> ...) to
`orders.events`, keyed by `order_id`. Then a Spark Structured Streaming job reads the
topic with `--available-now` (read everything new, then stop) and writes Parquet.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
import lab_runner
assert lab_runner.kafka_reachable(), "Kafka isn't running - make kafka-install"

In [ ]:
lab.produce(orders=20, run_id=RID)

In [ ]:
ev = lab.topic('orders.events', last=200, run_id=RID)
ev[['order_id', 'event_type', 'event_time', '_partition', '_offset']].head(10)

In [ ]:
ev.groupby('event_type').size().sort_values().plot.barh(title='events this run');

## Spark consumer -> Parquet bronze
Offsets are kept in the checkpoint, so running it again only reads new messages.

In [ ]:
r = lab.run('kafka-consumer-basic', run_id=RID)
lab.log_tail(r, 'bronze')

In [ ]:
lab.sql("SELECT * FROM parquet.`s3a://data/stream/bronze/kafka_events` LIMIT 5")

## Kafka -> Delta (exactly once) + Postgres status table

In [ ]:
r = lab.run('kafka-to-delta-pg', run_id=RID)
lab.log_tail(r, 'batch')

In [ ]:
lab.pg("SELECT status, count(*) FROM kafka_order_status WHERE run_id = %s GROUP BY 1 ORDER BY 2 DESC", (RID,))

In [ ]:
lab.sql("SELECT event_type, count(*) n FROM default.kafka_order_events GROUP BY 1 ORDER BY 2 DESC", lake=True)